##### Copyright 2026 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemma 3n - 使用Transformers.js 執行

作者：西塔姆‧梅爾
*   GitHub：[github.com/sitammeur](https://github.com/sitammeur/)
*   X：[@sitammeur](https://x.com/sitammeur)

描述：此 notebook 示範如何使用 Node.js 和 [Transformers.js](https://huggingface.co/docs/transformers.js/index) 在 Gemma 3n 模型上執行 inference。 Transformers.js 讓您直接在瀏覽器中執行Hugging Face 的變壓器模型，提供與Python 類似的JavaScript API。 它使用 ONNX Runtime 支援 NLP、電腦視覺、音訊和多模式任務，並允許輕鬆轉換 PyTorch、TensorFlow 和 JAX 模型。
<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Gemma/[Gemma_3n]Using_with_Transformersjs.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

## 設定

### 選擇 Colab runtime
要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 Gemma 3n 模型。在這種情況下，您可以使用 CPUruntime：
1. 在 Colab 視窗的右上角，選擇 **▾（其他連接選項）**。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **CPU**。

## 安裝

讓我們開始安裝依賴項。

In [ ]:
# Install Node.js
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs

## 建立 Node.js 項目

建立一個新的 Node.js 專案並透過 [NPM](https://www.npmjs.com/package/@huggingface/transformers) 安裝所需的轉換器套件。

In [ ]:
# Create project directory
!mkdir gemma3n-node
%cd gemma3n-node

# Initialize NPM project
!npm init -y
!npm i @huggingface/transformers wavefile

In [ ]:
%%writefile package.json

{
  "name": "gemma3n-node",
  "version": "1.0.0",
  "main": "index.js",
  "type": "module",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": "",
  "dependencies": {
    "@huggingface/transformers": "^3.8.1",
    "wavefile": "^11.0.0"
  }
}

## Transformers.js 推論

現在，讓我們使用 Transformers.js 在 Gemma 3n 型號上執行 inference。首先，創建圖像和文字、音訊和文字、圖像和音訊的生成管道。然後，準備輸入以執行 inference 並獲得所需的輸出。作為參考，您可以在 ONNX 模型部分下的 Hugging Face 模型中心查看模型頁面[此處](https://huggingface.co/onnx-community/gemma-3n-E2B-it-ONNX)。

### 圖片+文字推論

In [ ]:
# Show the image from the URL
from PIL import Image
import requests

url = "https://jethac.github.io/assets/juice.jpg"
img = Image.open(requests.get(url, stream=True).raw)
img

In [ ]:
%%writefile index.js

// Import the required modules
import {
  AutoProcessor,
  AutoModelForImageTextToText,
  load_image,
} from "@huggingface/transformers";

// Load processor and model
const model_id = "onnx-community/gemma-3n-E2B-it-ONNX";
const processor = await AutoProcessor.from_pretrained(model_id);
const model = await AutoModelForImageTextToText.from_pretrained(model_id, {
  dtype: {
    embed_tokens: "q8",
    audio_encoder: "q8",
    vision_encoder: "fp16",
    decoder_model_merged: "q4",
  },
  device: "cpu",
});

// Define the list of messages
const messages = [
  {
    role: "user",
    content: [
      { type: "image" },
      { type: "text", text: "Describe this image in detail." },
    ],
  },
];

try {
  // Prepare prompt
  const prompt = processor.apply_chat_template(messages, {
    add_generation_prompt: true,
  });

  // Prepare inputs
  const url = "https://jethac.github.io/assets/juice.jpg";
  const image = await load_image(url);
  const audio = null;

  const inputs = await processor(prompt, image, audio, {
    add_special_tokens: false,
  });

  // Generate output
  const outputs = await model.generate({
    ...inputs,
    max_new_tokens: 512,
    do_sample: false,
  });

  // Decode output
  const promptLen = inputs.input_ids.dims.at(-1);
  const decoded = processor.batch_decode(outputs.slice(null, [promptLen, null]), {
    skip_special_tokens: true,
  });

  console.log(decoded[0]);
} catch (error) {
  console.error("Error generating response:", error);
}

In [ ]:
# Run the node.js application (Image + Text)
!node index.js

### 音頻+文字推論

In [ ]:
# Display audio
from IPython.display import Audio
import requests

url = "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/jfk.wav"

audio_bytes = requests.get(url).content
Audio(audio_bytes)

In [ ]:
%%writefile index.js

// Import the required modules
import {
  AutoProcessor,
  AutoModelForImageTextToText,
} from "@huggingface/transformers";
import wavefile from "wavefile";

// Load processor and model
const model_id = "onnx-community/gemma-3n-E2B-it-ONNX";
const processor = await AutoProcessor.from_pretrained(model_id);
const model = await AutoModelForImageTextToText.from_pretrained(model_id, {
  dtype: {
    embed_tokens: "q8",
    audio_encoder: "q4",
    vision_encoder: "fp16",
    decoder_model_merged: "q4",
  },
  device: "cpu",
});

// Define the list of messages
const messages = [
  {
    role: "user",
    content: [
      { type: "audio" },
      { type: "text", text: "Transcribe this audio." },
    ],
  },
];

try {
  // Prepare prompt
  const prompt = processor.apply_chat_template(messages, {
    add_generation_prompt: true,
  });

  // Prepare inputs (audio from URL -> Float32Array @ model sample rate)
  const url =
    "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/jfk.wav";

  const buffer = Buffer.from(await fetch(url).then((x) => x.arrayBuffer()));
  const wav = new wavefile.WaveFile(buffer);

  // Pipeline expects Float32 samples
  wav.toBitDepth("32f");
  wav.toSampleRate(processor.feature_extractor.config.sampling_rate);

  let audioData = wav.getSamples();

  // Convert stereo -> mono (simple average with sqrt(2) normalization, like the docs)
  if (Array.isArray(audioData)) {
    if (audioData.length > 1) {
      for (let i = 0; i < audioData[0].length; ++i) {
        audioData[0][i] =
          (Math.sqrt(2) * (audioData[0][i] + audioData[1][i])) / 2;
      }
    }
    audioData = audioData[0];
  }

  const image = null;
  const audio = audioData;

  const inputs = await processor(prompt, image, audio, {
    add_special_tokens: false,
  });

  // Generate output
  const outputs = await model.generate({
    ...inputs,
    max_new_tokens: 512,
    do_sample: false,
  });

  // Decode output
  const promptLen = inputs.input_ids.dims.at(-1);
  const decoded = processor.batch_decode(outputs.slice(null, [promptLen, null]), {
    skip_special_tokens: true,
  });

  console.log(decoded[0]);
} catch (error) {
  console.error("Error generating response:", error);
}

In [ ]:
# Run the node.js application (Audio + Text)
!node index.js

### 影像+音頻推論

In [ ]:
# Show the image from the URL
from PIL import Image
import requests

url = "https://jethac.github.io/assets/juice.jpg"
img = Image.open(requests.get(url, stream=True).raw)
img

In [ ]:
# Display audio
from IPython.display import Audio
import requests

url = "https://raw.githubusercontent.com/sitammeur/test-assets/main/cat.wav"

audio_bytes = requests.get(url).content
Audio(audio_bytes)

In [ ]:
%%writefile index.js

// Import the required modules
import {
  AutoProcessor,
  AutoModelForImageTextToText,
  load_image,
} from "@huggingface/transformers";
import wavefile from "wavefile";

// Load processor and model
const model_id = "onnx-community/gemma-3n-E2B-it-ONNX";
const processor = await AutoProcessor.from_pretrained(model_id);
const model = await AutoModelForImageTextToText.from_pretrained(model_id, {
  dtype: {
    embed_tokens: "q8",
    audio_encoder: "q4",
    vision_encoder: "fp16",
    decoder_model_merged: "q4",
  },
  device: "cpu",
});

// Define the list of messages
const messages = [
  {
    role: "user",
    content: [{ type: "image" }, { type: "audio" }],
  },
];

try {
  // Prepare prompt
  const prompt = processor.apply_chat_template(messages, {
    add_generation_prompt: true,
  });

  // Prepare inputs (image + audio from URLs)
  const imageUrl = "https://jethac.github.io/assets/juice.jpg";
  const image = await load_image(imageUrl);

  const audioUrl = "https://raw.githubusercontent.com/sitammeur/test-assets/main/cat.wav";
  const buffer = Buffer.from(await fetch(audioUrl).then((r) => r.arrayBuffer()));
  const wav = new wavefile.WaveFile(buffer);

  // Pipeline expects Float32 samples at model sample rate
  wav.toBitDepth("32f");
  wav.toSampleRate(processor.feature_extractor.config.sampling_rate);

  let audioData = wav.getSamples();

  // Convert stereo -> mono
  if (Array.isArray(audioData) && audioData.length > 1) {
    const left = audioData[0];
    const right = audioData[1];
    audioData = left.map((_, i) => (Math.sqrt(2) * (left[i] + right[i])) / 2);
  } else if (Array.isArray(audioData)) {
    // If it's an array wrapper (single channel), unwrap it
    audioData = audioData[0];
  }

  const inputs = await processor(prompt, image, audioData, {
    add_special_tokens: false,
  });

  // Generate output
  const outputs = await model.generate({
    ...inputs,
    max_new_tokens: 512,
    do_sample: false,
  });

  // Decode output
  const promptLen = inputs.input_ids.dims.at(-1);
  const decoded = processor.batch_decode(outputs.slice(null, [promptLen, null]), {
    skip_special_tokens: true,
  });

  console.log(decoded[0]);
} catch (error) {
  console.error("Error generating response:", error);
}

In [ ]:
# Run the node.js application (Image + Audio)
!node index.js

## 結論

恭喜！您已透過 Node.js 環境使用 Transformers.js 在 Gemma 3n 模型上成功執行 inference。現在您可以將其整合到您的專案中。